# VaaniRAG Offline Ingestion Pipeline

This notebook is the user execution interface for the **VaaniRAG Multilingual Offline Ingestion Pipeline**.
It processes English, Hindi, and Marathi text from the Hugging Face `ai4bharat/MSMARCO-XI` dataset, cleans and deduplicates the passages, partitions them using custom chunking strategies, encodes them locally using BAAI's `bge-m3` model on a GPU, validates vectors, and uploads them to a Pinecone vector database.

### Pipeline Steps:
1. **Environment Setup & GPU Check**: Verify hardware resources.
2. **Configuration & Secrets**: Securely load Pinecone credentials.
3. **Schema & Dataset Inspection**: Analyze HF stream records.
4. **Embedding Verification**: Run tests on the local `bge-m3` instance.
5. **100-Row Dry Run**: Execute the modular pipeline on 100 rows per language (no cloud costs).
6. **Strategy Benchmarking**: Compare original, sentence, fixed, semantic, and adaptive chunkers.
7. **Pinecone Live Upload**: Deploy verified vectors to Pinecone Cloud namespaces.

## Step 1: Install Dependencies
First, we install all required packages.

In [ ]:
# Install pipeline packages
!pip install datasets sentence-transformers pinecone-client pydantic python-dotenv tqdm pytest torch numpy

## Step 2: GPU Check
The BGE-M3 embedding step requires a GPU (T4 in Colab) for performance. We check PyTorch CUDA support.

In [ ]:
import torch

gpu_available = torch.cuda.is_available()
print("GPU Available:", gpu_available)
if gpu_available:
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)
else:
    print("WARNING: CUDA is unavailable. Pipeline will fall back to CPU (slow).")

## Step 3: Configure Credentials (Colab Secrets)

To avoid hardcoding keys, configure `PINECONE_API_KEY` in the **Secrets** tab (key icon in the left sidebar) and enable notebook access. The code below retrieves it securely.

In [ ]:
import os

try:
    from google.colab import userdata
    # Get key from Colab Secrets manager
    pinecone_key = userdata.get('PINECONE_API_KEY')
    os.environ['PINECONE_API_KEY'] = pinecone_key
    print("SUCCESS: Pinecone API Key loaded from Colab Secrets.")
except Exception as e:
    print("Colab Secrets not detected (running locally or key not configured). Checking local environment variables...")
    if 'PINECONE_API_KEY' in os.environ:
        print("SUCCESS: PINECONE_API_KEY found in environment.")
    else:
        print("WARNING: PINECONE_API_KEY is not set. Live Pinecone upload will fail.")

## Step 4: Run Dataset Schema Inspection
We inspect columns, field types, and text lengths for all languages.

In [ ]:
!python scripts/inspect_dataset.py

## Step 5: Test Local Embeddings (BGE-M3)
We verify local model loading, output dimension correctness (1024), and compute throughput.

In [ ]:
!python scripts/test_embedding.py

## Step 6: Execute 100-Row Dry Run
Run the full modular pipeline on a safe limit of 100 rows per language with Pinecone uploads disabled (`--dry-run`).

In [ ]:
!python -m ingestion.pipeline --languages en,hi,mr --max-rows 100 --strategy adaptive --dry-run

## Step 7: Benchmark Chunking Strategies
Compare chunk counts, average token distribution, and local vector storage requirements for all five strategies.

In [ ]:
!python scripts/benchmark_ingestion.py

## Step 8: Live Upload (Pinecone Cloud Ingestion)

After successful dry run verification, trigger the upload process by enabling `--upload`.
This connects to Pinecone Cloud, ensures the `vaani-rag` index matches specifications (1024-dim, cosine), and uploads records into `en`, `hi`, and `mr` namespaces.

In [ ]:
# To trigger a live upload, uncomment and run this line:
# !python -m ingestion.pipeline --languages en,hi,mr --max-rows 100 --strategy adaptive --upload
print("Ensure PINECONE_API_KEY is active in secrets before running live uploads.")